# 05. 用PyTorch实现线性回归

用pytorch的工具解决线性模型

之前重点要人工算gradient，现在重点是构造张量的计算图

只要能成功制造出来，之后会自动算出梯度

PyTorch Fashion(风格)

1、prepare dataset

2、design model using Class  # 目的是为了前向传播forward，即计算y hat(预测值)

3、Construct loss and optimizer (using PyTorch API) 其中，计算loss是为了进行反向传播，optimizer是为了更新梯度。

4、Training cycle (forward,backward,update)


代码说明：

1、Module 实现了魔法函数__call__()，call()里面有一条语句是要调用forward()。因此新写的类中需要重写forward()覆盖掉父类中的forward()

2、call函数的另一个作用是可以直接在对象后面加()，例如实例化的model对象，和实例化的linear对象

3、本算法的forward体现是通过以下语句实现的：

y_pred = model(x_data)

由于魔法函数call的实现,model(x_data)将会调用model.forward(x_data)函数，model.forward(x_data)函数中的

y_pred = self.linear(x)

self.linear(x)也由于魔法函数call的实现将会调用torch.nn.Linear类中的forward，至此完成封装，也就是说forward最终是在torch.nn.Linear类中实现的，具体怎么实现，可以不用关心，就是y= wx + b。

4、本算法的反向传播，计算梯度是通过以下语句实现的：

loss.backward() # 反向传播，计算梯度

5、本算法的参数(w,b)更新，是通过以下语句实现的：

optimizer.step() # update 参数，即更新w和b的值

6、 每一次epoch的训练过程，总结就是

①前向传播，求y hat （输入的预测值）

②根据y_hat和y_label(y_data)计算loss

③反向传播 backward (计算梯度)

④根据梯度，更新参数

7、本实例是批量数据处理，不要被optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)误导了，以为见了SGD就是随机梯度下降。要看传进来的数据是单个的还是批量的。这里的x_data是3个数据，是一个batch ，调用的PyTorch API是 torch.optim.SGD，但这里的SGD不是随机梯度下降，而是批量梯度下降。也就是说，梯度下降算法使用的是随机梯度下降，还是批量梯度下降，还是mini-batch梯度下降，用的API都是 torch.optim.SGD。

8、torch.nn.MSELoss也跟torch.nn.Module有关，参与计算图的构建，torch.optim.SGD与torch.nn.Module无关，不参与构建计算图。

In [2]:
import  torch

# prepare dataset
# x,y是矩阵，3行1列 也就是说总共有3个数据，每个数据只有1个特征
x_data = torch.Tensor([[1.0],[2.0],[3.0]])
y_data = torch.Tensor([[2.0],[4.0],[6.0]])


#design model using class
"""
our model class should be inherit from nn.Module, which is base class for all neural network modules.
member methods __init__() and forward() have to be implemented
class nn.linear contain two member Tensors: weight and bias
class nn.Linear has implemented the magic method __call__(),which enable the instance of the class can
be called just like a function.Normally the forward() will be called 
"""
class LinearModel(torch.nn.Module):
    def __init__(self): #构造函数
        super(LinearModel,self).__init__()
        #构造对象，并说明输入输出的维数，第三个参数默认为true，表示用到b
        # (1,1)是指输入x和输出y的特征维度，这里数据集中的x和y的特征都是1维的
        # 该线性层需要学习的参数是w和b  获取w/b的方式分别是~linear.weight/linear.bias
        self.linear = torch.nn.Linear(1,1)
    def forward(self, x):
        y_pred = self.linear(x) #可调用对象，计算y=wx+b
        return  y_pred

model = LinearModel()#实例化模型

# construct loss and optimizer
# criterion = torch.nn.MSELoss(size_average = False)
criterion = torch.nn.MSELoss(size_average=False)
optimizer = torch.optim.SGD(model.parameters(),lr=0.01) #model.parameters()会扫描module中的所有成员，如果成员中有相应权重，那么都会将结果加到要训练的参数集合上


# training cycle forward, backward, update
for epoch in range(1000):
    y_pred = model(x_data) # forward:predict
    loss = criterion(y_pred, y_data) # forward: loss
    print(epoch, loss.item())
 
    optimizer.zero_grad() # the grad computer by .backward() will be accumulated. so before backward, remember set the grad to zero
    loss.backward() # backward: autograd，自动计算梯度
    optimizer.step() # update 参数，即更新w和b的值

print('w=',model.linear.weight.item())
print('b=',model.linear.bias.item())

x_test = torch.Tensor([[4.0]])
y_test = model(x_test)
print('y_pred = ', y_test.data)


/home/tony/anaconda3/envs/pytorch-gpu/lib/python3.10/site-packages/torch/nn/_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


0 86.70188903808594
1 38.91517639160156
2 17.637325286865234
3 8.16052532196045
4 3.9372849464416504
5 2.0528430938720703
6 1.2096303701400757
7 0.8300056457519531
8 0.6568179130554199
9 0.5755900740623474
10 0.5353599190711975
11 0.5134388208389282
12 0.49972715973854065
13 0.48972567915916443
14 0.48143255710601807
15 0.47395455837249756
16 0.4668946862220764
17 0.46007347106933594
18 0.45341217517852783
19 0.44687420129776
20 0.44044196605682373
21 0.4341079592704773
22 0.42786705493927
23 0.4217170178890228
24 0.41565605998039246
25 0.4096822440624237
26 0.4037945866584778
27 0.3979911804199219
28 0.3922712802886963
29 0.3866337239742279
30 0.38107746839523315
31 0.375600665807724
32 0.37020277976989746
33 0.36488237977027893
34 0.3596383333206177
35 0.3544698655605316
36 0.3493754267692566
37 0.3443543612957001
38 0.33940523862838745
39 0.33452731370925903
40 0.3297199308872223
41 0.3249812722206116
42 0.3203108310699463
43 0.3157072961330414
44 0.3111703395843506
45 0.30669832229

不同优化器，他们的性能在使用上有什么区别？
以下包含了Adagrad Adam adamax ASGD RMSprop Rprop SGD七种优化器的loss下降图。其实还有一种优化器LBFGS